In [10]:
import duckdb
import sqlite3
import os
import pandas as pd
import numpy as np

### SQLite から DuckDB へ移行

In [9]:
print("DuckDB version:", duckdb.__version__)

DuckDB version: 1.5.3


In [2]:
# SQLiteからDuckDBへ移行（一度だけ）
con = duckdb.connect("test_db/homogenous.duckdb")

In [ ]:
sqlite_con = sqlite3.connect('test_db/homogenous.sqlite3')

# チャンクごとに処理（メモリ節約）
chunk_size = 500_000
offset = 0
first = True

while True:
    df = pd.read_sql(
        f"SELECT * FROM NullFilled_DropOrigin_OneHotEncoded_homogenous LIMIT {chunk_size} OFFSET {offset}",
        sqlite_con
    )
    if df.empty:
        break

    df["Date"] = pd.to_datetime(df["Date"]) # なぜか Date列が NUM型 になっていたため、日付型に変換してからDuckDBにロードする -> sqlite3 に日付型がないため

    if first:
        con.execute("CREATE TABLE homogenous AS SELECT * FROM df")
        first = False
    else:
        con.execute("INSERT INTO homogenous SELECT * FROM df")

    offset += chunk_size
    print(f"Loaded {offset:,} rows...")

sqlite_con.close()
print("Done.")

Loaded 500,000 rows...
Loaded 1,000,000 rows...
Loaded 1,500,000 rows...
Loaded 2,000,000 rows...
Loaded 2,500,000 rows...
Loaded 3,000,000 rows...
Loaded 3,500,000 rows...
Loaded 4,000,000 rows...
Loaded 4,500,000 rows...
Loaded 5,000,000 rows...
Loaded 5,500,000 rows...
Loaded 6,000,000 rows...
Loaded 6,500,000 rows...
Loaded 7,000,000 rows...
Loaded 7,500,000 rows...
Loaded 8,000,000 rows...
Loaded 8,500,000 rows...
Loaded 9,000,000 rows...
Loaded 9,500,000 rows...
Loaded 10,000,000 rows...
Loaded 10,500,000 rows...
Loaded 11,000,000 rows...
Loaded 11,500,000 rows...
Loaded 12,000,000 rows...
Loaded 12,500,000 rows...
Loaded 13,000,000 rows...
Loaded 13,500,000 rows...
Loaded 14,000,000 rows...
Loaded 14,500,000 rows...
Loaded 15,000,000 rows...
Loaded 15,500,000 rows...
Loaded 16,000,000 rows...
Done.


### 移行完了

以降は，

<code>$ duckdb -ui</code> 

より，SQLを発行可能

In [14]:
con.close()